# Inventory Diff and Reference Generation

Runs inventory change detection and per-object reference generation in one notebook task.

In [ ]:
import sys
from pathlib import Path

repo_root = Path('..').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from utils.config_utils import check_runtime_readiness, load_pipeline_config, resolve_secrets
from pipeline.inventory import build_inventory_snapshot_and_diff
from pipeline.generate_parquet import generate_references_in_parallel, save_ledger_after_success

import virtualizarr
import obstore
import obspec_utils
print('Imports OK')

In [ ]:
check_runtime_readiness()
kp= load_pipeline_config("../configs/config.yaml")
ACCESS_KEY, SECRET_KEY = resolve_secrets(kp, dbutils)

## MasterLedger building

In [ ]:
ledger = build_inventory_snapshot_and_diff(
    kp=kp,
    access_key=ACCESS_KEY,
    secret_key=SECRET_KEY,
)

print('Inventory summary:', ledger['summary'])
print('Sample new keys:', ledger['diff']['new'][:10])
print('Sample changed keys:', ledger['diff']['changed'][:10])
print('Sample deleted keys:', ledger['diff']['deleted'][:10])

inventory_diff = ledger['diff']
inventory_objects = ledger['current_objects']
pending_ledger = ledger['next_ledger']

## Reference Generation Run Mode

Use Cell 7 to choose run mode:
- Dry run: generate references for one object only (verification).
- Full run: process all new/changed objects and capture runtime metrics.

In [ ]:
from datetime import datetime, UTC

# Toggle run mode here before execution.
DRY_RUN = True
FULL_RUN = not DRY_RUN

run_meta = {
    "mode": "dry-run" if DRY_RUN else "full-run",
    "selected_key": None,
    "started_at_utc": None,
    "ended_at_utc": None,
    "duration_seconds": 0.0,
}

if DRY_RUN:
    candidate_keys = sorted(set(inventory_diff.get("new", []) + inventory_diff.get("changed", [])))
    if not candidate_keys:
        reference_generation = {
            "summary": {
                "scanned": len(inventory_objects),
                "changed_or_new": 0,
                "generated": 0,
                "skipped": 0,
                "failed": 0,
            },
            "results": [],
            "failures": [],
        }
        print("Dry-run: no new/changed objects to test.")
    else:
        test_key = candidate_keys[0]
        run_meta["selected_key"] = test_key
        print(f"Dry-run testing single key: {test_key}")

        dry_inventory_diff = {"new": [test_key], "changed": [], "deleted": [], "unchanged": []}
        dry_current_objects = {test_key: inventory_objects[test_key]}

        started_at = datetime.now(UTC)
        reference_generation = generate_references_in_parallel(
            kp=kp,
            access_key=ACCESS_KEY,
            secret_key=SECRET_KEY,
            inventory_diff=dry_inventory_diff,
            current_objects=dry_current_objects,
        )
        ended_at = datetime.now(UTC)

        run_meta["started_at_utc"] = started_at.isoformat()
        run_meta["ended_at_utc"] = ended_at.isoformat()
        run_meta["duration_seconds"] = round((ended_at - started_at).total_seconds(), 2)
else:
    started_at = datetime.now(UTC)
    reference_generation = generate_references_in_parallel(
        kp=kp,
        access_key=ACCESS_KEY,
        secret_key=SECRET_KEY,
        inventory_diff=inventory_diff,
        current_objects=inventory_objects,
    )
    ended_at = datetime.now(UTC)

    run_meta["started_at_utc"] = started_at.isoformat()
    run_meta["ended_at_utc"] = ended_at.isoformat()
    run_meta["duration_seconds"] = round((ended_at - started_at).total_seconds(), 2)

print("Run mode:", run_meta["mode"])
print("Reference generation summary:", reference_generation["summary"])
print("Generated sample:", [r["key"] for r in reference_generation["results"] if r["status"] == "generated"][:10])
print("Failures:", reference_generation["failures"][:5])
print("Duration (seconds):", run_meta["duration_seconds"])

## Commit Ledger and Log Runtime

- Full run: commits ledger when no failures and appends runtime metrics to Delta.
- Dry run: skips ledger commit and runtime logging.

In [ ]:
from datetime import datetime, UTC
from uuid import uuid4

runtime_log_path = "/Volumes/workspace/weather/kerchunk/runtime_log/"

if FULL_RUN:
    if reference_generation["summary"]["failed"] == 0:
        save_ledger_after_success(
            ledger_path=kp["output"]["ledger_path"],
            next_ledger=pending_ledger,
            generation_summary=reference_generation["summary"],
        )
        commit_status = "committed"
        print("Ledger committed:", kp["output"]["ledger_path"])
    else:
        commit_status = "blocked_due_to_failures"
        print("Ledger NOT committed due to generation failures.")

    # Capture end-of-pipeline time after commit decision.
    pipeline_ended_at = datetime.now(UTC)
    total_duration_seconds = run_meta["duration_seconds"]
    if run_meta["started_at_utc"]:
        total_duration_seconds = round(
            (pipeline_ended_at - datetime.fromisoformat(run_meta["started_at_utc"])).total_seconds(),
            2,
        )

    runtime_log_row = {
        "run_id": str(uuid4()),
        "mode": run_meta["mode"],
        "started_at_utc": run_meta["started_at_utc"],
        "ended_at_utc": pipeline_ended_at.isoformat(),
        "duration_seconds": float(total_duration_seconds),
        "scanned": int(reference_generation["summary"].get("scanned", 0)),
        "changed_or_new": int(reference_generation["summary"].get("changed_or_new", 0)),
        "generated": int(reference_generation["summary"].get("generated", 0)),
        "failed": int(reference_generation["summary"].get("failed", 0)),
        "commit_status": commit_status,
    }

    dbutils.fs.mkdirs(runtime_log_path)
    spark.createDataFrame([runtime_log_row]).write.format("delta").mode("append").save(runtime_log_path)
    print("Runtime log appended to Delta path:", runtime_log_path)
    print("Full-run duration (seconds):", runtime_log_row["duration_seconds"])
else:
    print("Dry-run mode: ledger commit skipped and runtime log not written.")
    if reference_generation["summary"].get("failed", 0) == 0:
        print("Dry-run verification passed.")
    else:
        print("Dry-run verification failed. Inspect failures above.")